In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load
df = pd.read_csv("../data/train.csv")    

# 2. Clean
df = df.drop(columns=["Id"])
df = df[~((df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000))]   
df["MSSubClass"] = df["MSSubClass"].astype(str)   

# 3. Features (X) and target (y, log transform)
X = df.drop(columns=["SalePrice"])
y = np.log1p(df["SalePrice"])

# 4. Train / test split (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test: ", X_test.shape)

Train: (1166, 79)
Test:  (292, 79)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(exclude=np.number).columns

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

# compare two models
models = {
    "Ridge": Ridge(alpha=10),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
}

for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f"{name}: RMSE(log) = {rmse:.4f}, R2 = {r2:.4f}")

Ridge: RMSE(log) = 0.1290, R2 = 0.9013
RandomForest: RMSE(log) = 0.1436, R2 = 0.8777


In [3]:
pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 594.3 kB/s eta 0:01:22
    --------------------------------------- 0.8/48.9 MB 756.6 kB/s eta 0:01:04
    --------------------------------------- 1.0/48.9 MB 918.1 kB/s eta 0:00:53
   - -------------------------------------- 1.6/48.9 MB


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBRegressor

models = {
    "Ridge": Ridge(alpha=10),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=3,
                            subsample=0.8, colsample_bytree=0.8, random_state=42),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    scores = cross_val_score(pipe, X, y, cv=kf, scoring="neg_root_mean_squared_error")
    rmse = -scores
    print(f"{name}: RMSE(log) = {rmse.mean():.4f} (+/- {rmse.std():.4f})")
    

Ridge: RMSE(log) = 0.1247 (+/- 0.0084)
RandomForest: RMSE(log) = 0.1389 (+/- 0.0094)
XGBoost: RMSE(log) = 0.1172 (+/- 0.0085)


In [5]:
import os
import joblib

# Final pipeline (XGBoost)
final_model = Pipeline([
    ("prep", preprocess),
    ("model", XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=3,
                           subsample=0.8, colsample_bytree=0.8, random_state=42)),
])

# Ella data-vaiyum vachu train
final_model.fit(X, y)

# Save
os.makedirs("../models", exist_ok=True)
joblib.dump(final_model, "../models/house_price_model.pkl")
print("Model saved!")

# Test: save aana model-a load panni, oru veedu predict pannurom
loaded = joblib.load("../models/house_price_model.pkl")
sample = X.iloc[[0]]                          # mudhal veedu
pred_log = loaded.predict(sample)
pred_price = np.expm1(pred_log)[0]            # log-la irundhu real price-ku
actual = np.expm1(y.iloc[0])

print(f"Predicted: {pred_price:,.0f}")
print(f"Actual:    {actual:,.0f}")

Model saved!
Predicted: 204,355
Actual:    208,500


In [6]:
num_defaults = X.select_dtypes(include=np.number).median()
cat_defaults = X.select_dtypes(exclude=np.number).mode().iloc[0]

defaults = {**num_defaults.to_dict(), **cat_defaults.to_dict()}
joblib.dump(defaults, "../models/defaults.pkl")
print("Defaults saved:", len(defaults), "columns")

Defaults saved: 79 columns
